In [1]:
# Experiment 2: Production Rules and the Match Phase

# Each rule contains:
# - id: rule identifier
# - conditions: facts required in working memory
# - conclusion: fact produced when the rule fires

rules = [
    {
        "id": "R1",
        "conditions": ["fever", "cough"],
        "conclusion": "flu"
    },
    {
        "id": "R2",
        "conditions": ["flu", "body_ache"],
        "conclusion": "viral_infection"
    },
    {
        "id": "R3",
        "conditions": ["fever", "sore_throat"],
        "conclusion": "throat_infection"
    },
    {
        "id": "R4",
        "conditions": ["cough", "breathing_problem"],
        "conclusion": "respiratory_problem"
    },
    {
        "id": "R5",
        "conditions": ["viral_infection"],
        "conclusion": "take_rest"
    }
]

# Initial Working Memory
WM = {
    "fever",
    "cough",
    "body_ache"
}

print("Production Rules:")
for rule in rules:
    print(
        f"{rule['id']}: IF "
        f"{' AND '.join(rule['conditions'])} "
        f"THEN {rule['conclusion']}"
    )

print("\nWorking Memory:")
print(WM)

Production Rules:
R1: IF fever AND cough THEN flu
R2: IF flu AND body_ache THEN viral_infection
R3: IF fever AND sore_throat THEN throat_infection
R4: IF cough AND breathing_problem THEN respiratory_problem
R5: IF viral_infection THEN take_rest

Working Memory:
{'fever', 'body_ache', 'cough'}


In [6]:
def match_rules(rules, WM):
    """
    Match every production rule against the current
    working memory.

    A rule is applicable when:
    1. All its conditions are present in WM.
    2. Its conclusion is not already present in WM.
    """

    conflict_set = []

    for rule in rules:

        # Check whether every condition of the rule
        # is available in working memory.
        conditions_satisfied = all(
            condition in WM
            for condition in rule["conditions"]
        )

        # Refractoriness:
        # Do not select a rule if its conclusion
        # is already present in working memory.
        conclusion_not_present = (
            rule["conclusion"] not in WM
        )

        if conditions_satisfied and conclusion_not_present:
            conflict_set.append(rule["id"])

    return conflict_set


# Run the match phase
conflict_set = match_rules(rules, WM)

print("Working Memory:")
print(WM)

print("\nConflict Set:")
print(conflict_set)

Working Memory:
{'fever', 'body_ache', 'cough'}

Conflict Set:
['R1']


In [7]:
# Exercise 1:
# Add facts to Working Memory one at a time
# and observe how the conflict set changes.

WM_experiment = set()

facts_to_add = [
    "fever",
    "cough",
    "body_ache",
    "flu",
    "sore_throat",
    "breathing_problem"
]

print("Changes in Conflict Set\n")
print("-" * 50)

for fact in facts_to_add:

    # Add one new fact to working memory
    WM_experiment.add(fact)

    # Perform the match phase
    conflict_set = match_rules(rules, WM_experiment)

    print(f"\nAdded Fact: {fact}")
    print(f"Working Memory: {WM_experiment}")
    print(f"Conflict Set: {conflict_set}")

Changes in Conflict Set

--------------------------------------------------

Added Fact: fever
Working Memory: {'fever'}
Conflict Set: []

Added Fact: cough
Working Memory: {'fever', 'cough'}
Conflict Set: ['R1']

Added Fact: body_ache
Working Memory: {'fever', 'body_ache', 'cough'}
Conflict Set: ['R1']

Added Fact: flu
Working Memory: {'fever', 'flu', 'body_ache', 'cough'}
Conflict Set: ['R2']

Added Fact: sore_throat
Working Memory: {'cough', 'fever', 'sore_throat', 'body_ache', 'flu'}
Conflict Set: ['R2', 'R3']

Added Fact: breathing_problem
Working Memory: {'cough', 'fever', 'sore_throat', 'breathing_problem', 'body_ache', 'flu'}
Conflict Set: ['R2', 'R3', 'R4']


In [8]:
import time

# ---------------------------------------------------------
# Exercise 2: Support for NOT conditions
# ---------------------------------------------------------

def match_rules_with_not(rules, WM):
    """
    Supports normal conditions and negated conditions.

    Example:
        "NOT headache"

    means that 'headache' must NOT be present in WM.
    """

    conflict_set = []

    for rule in rules:
        conditions_satisfied = True

        for condition in rule["conditions"]:

            if condition.startswith("NOT "):
                fact = condition[4:]

                if fact in WM:
                    conditions_satisfied = False
                    break

            else:
                if condition not in WM:
                    conditions_satisfied = False
                    break

        if (
            conditions_satisfied
            and rule["conclusion"] not in WM
        ):
            conflict_set.append(rule["id"])

    return conflict_set


# Rule containing a NOT condition
not_rules = rules + [
    {
        "id": "R6",
        "conditions": ["fever", "NOT headache"],
        "conclusion": "mild_condition"
    }
]

WM_not = {
    "fever",
    "cough"
}

print("Exercise 2: NOT Condition")
print("-------------------------")
print("Working Memory:", WM_not)

conflict_set = match_rules_with_not(
    not_rules,
    WM_not
)

print("Conflict Set:", conflict_set)


# ---------------------------------------------------------
# Exercise 3: Performance with 1,000 Rules
# ---------------------------------------------------------

def generate_rules(number_of_rules):
    """
    Automatically generate production rules
    for performance testing.
    """

    generated_rules = []

    for i in range(number_of_rules):
        generated_rules.append({
            "id": f"R{i + 1}",
            "conditions": [
                f"fact_{i % 100}",
                f"fact_{(i + 1) % 100}"
            ],
            "conclusion": f"conclusion_{i}"
        })

    return generated_rules


# Generate 1,000 rules
large_rule_base = generate_rules(1000)

# Working memory containing 100 facts
large_WM = {
    f"fact_{i}"
    for i in range(100)
}

# Measure matching time
start_time = time.perf_counter()

large_conflict_set = match_rules(
    large_rule_base,
    large_WM
)

end_time = time.perf_counter()

match_time = end_time - start_time

print("\nExercise 3: Performance Test")
print("-----------------------------")
print("Number of Rules:", len(large_rule_base))
print("Number of Facts:", len(large_WM))
print("Number of Matching Rules:", len(large_conflict_set))
print(f"Match Time: {match_time:.8f} seconds")

Exercise 2: NOT Condition
-------------------------
Working Memory: {'fever', 'cough'}
Conflict Set: ['R1', 'R6']

Exercise 3: Performance Test
-----------------------------
Number of Rules: 1000
Number of Facts: 100
Number of Matching Rules: 1000
Match Time: 0.00072550 seconds
